In [1]:
%load_ext autoreload
%autoreload 2

from tensorboard.backend.event_processing import event_accumulator
import os
import scipy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt
import copy
from tqdm import tqdm
import csv
from scipy.special import softmax
device = torch.device('cuda:0')
import random

import logging
logger = logging.getLogger(__name__)
logging.basicConfig(filename='train_log.log', level=logging.INFO)

from Dataset import ImportanceDataset, RealImportanceDataset, RealPredictionDataset, XBoxDatasetSimulation, Axios_ipsosdataset, HouseholdPulse_dataset
from GAN import GAN, WGAN_GP
from Discriminator import DataDiscriminator, DeepSetCritic
from util import set_seed
import itertools
import pandas as pd
#optimize over w directly, dont use any theta shenannigans. 
from sklearn import preprocessing
import pytorch_warmup as warmup
from DataProcessing import *
import pickle

In [ ]:
#d4p data processing

df = pd.read_stata("./data/progress_data/dfp_covid_tracking_poll.dta")
df = df[df['wave'] == 25]
columns_to_keep = []
columns_to_keep.append('nationalweight')
columns_to_keep.append('gender')
columns_to_keep.append('ethnicity')
columns_to_keep.append('education')
columns_to_keep.append('age')
columns_to_keep.append('region')
columns_to_keep.append('hhi') #household income
columns_to_keep.append('vax')

df = df[columns_to_keep].dropna(subset=columns_to_keep)


vax_binary = df['vax'].str.startswith('Yes,').astype(int)
weighted_avg = (vax_binary * df['nationalweight']).sum() / df['nationalweight'].sum()

#d4p procressing
df = pd.read_stata("./data/progress_data/dfp_covid_tracking_poll.dta")
df = df[df['wave'] == 25]

columns_to_keep = []
columns_to_keep.append('gender')
columns_to_keep.append('ethnicity')
columns_to_keep.append('education')
columns_to_keep.append('age')
columns_to_keep.append('region')
columns_to_keep.append('hhi') #household income
columns_to_keep.append('vax')

df = df[columns_to_keep].dropna(subset=columns_to_keep)

#d4p processing 2
def recode_gender(value):
    if value == 'Male':
        return 1
    elif value == 'Female':
        return 2
    else:
        return None
df['gender'] = df['gender'].apply(recode_gender)

def recode_census_age(value):
    if value >= 85: 
        return 5
    elif value >= 65:
        return 4
    elif value >= 50:
        return 3
    elif value >= 35:
        return 2
    elif value >= 18:
        return 1
df['age'] = df['age'].apply(recode_census_age)

def recode_census_region(value):
        if value == 1:
            return 1  # NorthEast
        elif value == 2:
            return 3  # MidWest
        elif value == 3:
            return 2  # South
        elif value == 4:
            return 4  # West
        else:
            return None  # Handle unexpected values
# Apply the recoding function to the region column in the census data
df['region'] = df['region'].apply(recode_census_region)

educ_mapping = {
            1: 1,  # Less than high school
            2: 2,  # High school graduate
            3: 3,  # Some college
            4: 3,  # Some college
            5: 3,  # Some college
            6: 4, # Bachelor's degree
            7: 5,  # Graduate degree
            8: 5,
            -3105: None,
        }
df['education'] = df['education'].map(educ_mapping)

race_map = {
        1: 1,  # White -> White, Alone
        2: 2,  # Black/African American -> Black, Alone
        3: 4,  # American Indian or Alaska Native -> Any other race alone, or race in combination
        5: 3,  # Chinese -> Asian, Alone
        7: 3,  # Japanese -> Asian, Alone
        4: 3,  # Other Asian or Pacific Islander -> Asian, Alone
        6: 3,  # Other Asian or Pacific Islander -> Asian, Alone
        8: 3,  # Other Asian or Pacific Islander -> Asian, Alone
        9: 3,  # Other Asian or Pacific Islander -> Asian, Alone
        10: 3,  # Other Asian or Pacific Islander -> Asian, Alone
        11: 3,  # Other Asian or Pacific Islander -> Asian, Alone
        12: 3,  # Other Asian or Pacific Islander -> Asian, Alone
        13: 3,  # Other Asian or Pacific Islander -> Asian, Alone
        14: 3,  # Other Asian or Pacific Islander -> Asian, Alone
        15: None,  # Other race, nec -> Any other race alone, or race in combination
        16: None,
    }
df['ethnicity'] = df['ethnicity'].map(race_map)

def map_income(value):
    if value == -3015:
        return None  # Question seen but not selected
    elif value <= 3:
        return 1  # Less than $25,000
    elif 4 <= value < 6:
        return 2  # $25,000 - $34,999
    elif 6 <= value < 9:
        return 3  # $35,000 - $49,999
    elif 9 <= value < 14:
        return 4  # $50,000 - $74,999
    elif 14 <= value < 19:
        return 5  # $75,000 - $99,999
    elif 19 <= value < 21:
        return 6  # $100,000 - $149,999
    elif 21 <= value < 23:
        return 7  # $150,000 - $199,999
    else:
        return 8  # $200,000 and above

df['hhi'] = df['hhi'].apply(map_income)

df = df.rename(columns={'gender': 'SEX'})
df = df.rename(columns={'education': 'EDUC'})
df = df.rename(columns={'region': 'REGION'})
df = df.rename(columns={'hhi': 'INCTOT'})
df = df.rename(columns={'ethnicity': 'RACE'})
df = df.rename(columns={'age': 'AGE'})

vax_binary = df['vax'].str.startswith('Yes,').astype(int)
df.drop('vax',axis=1)
df['vax'] = vax_binary

cols = list(df.columns)
cols.insert(0, cols.pop(cols.index('vax')))
df = df[cols]

print(df)

week='25'
df.to_csv('./data/progress_data/week'+week+'_cleaned.csv', index=False)

In [ ]:
#d4p processing continued 
from Dataset import D4P_dataset
EPOCHS = 100
DISC_LR = 1e-5
GT_LIMIT = 100 #25000
BIAS_LIMIT = 100 #1000
BATCH_SIZE =16
SUBSET_SIZE = 64
week='25'

seed=0
rngs = set_seed(seed,
                device,
                data_init=[True,True],
                data_gen =True,
                network_init=True,)

D1_rngs = copy.deepcopy(rngs)
D1_rngs['seed_bias'] = 359556
D2_rngs = copy.deepcopy(rngs)
D2_rngs['seed_bias'] = 280651
#load in both data sets

d = D4P_dataset(ground_truth_path='./data/censusHouseholdPulse_data/ipums_cleaned.csv',
            bias_path = './data/progress_data/week'+week+'_cleaned.csv',
            rngs=rngs,
            device=device,
            gt_limit = GT_LIMIT,
            )

In [ ]:
#analyzing the groupings of variables
#plotting events from saved run logs
# Path to the directory where SummaryWriter saved logs
import re
import ast
file_paths = ["runs_household_1_8_vars/"]

def isolate_variable_names(file_name):
    match = re.search(r"columns_to_keep:(\([^\)]*\))\|\|seed", file_name)
    if match:
        test = ast.literal_eval((match.group(1)))
        return test
    return None
def print_stats(runs_path):
    run_folders = []
    for name in os.listdir(runs_path):
        if os.path.isdir(os.path.join(runs_path,name)):
            run_folders.append(os.path.join(runs_path,name))

    event_files = []
    for run_folder in run_folders:
        for name in os.listdir(run_folder):
            if os.path.isdir(os.path.join(run_folder,name)):
                continue
            else:
                event_files.append(os.path.join(run_folder,name))

    all_vac_predictions = []
    all_l2_demographics = []
    all_names = []
    for i, event_file in enumerate(event_files):
        # Load event accumulator
        var_name_tuple = isolate_variable_names(event_file)
        if len(var_name_tuple) != num_vars:
            continue
        all_names.append(var_name_tuple)
        ea = event_accumulator.EventAccumulator(event_file)
        ea.Reload()
        #scalar_tags = ea.Tags().get('scalars', [])
        # List available tags (scalars, histograms, images, etc.)
        # Read scalar values (e.g., 'loss', 'accuracy')
        try:
            #demo_events = ea.Scalars("JS Divergence")
            demo_events = ea.Scalars("l2 norm demo diff")
            prediction_events = ea.Scalars("Vaccine prediction total") 
        except:
            print("error: ", event_file)
            continue
        #summ = 0
        #for iter, event in enumerate(prediction_events):
        #    print(event, l2_events[iter])
        cur_vac_pred = []
        cur_l2_demo = []
        for event in demo_events:
            cur_l2_demo.append(event.value)
        for event in prediction_events:
            cur_vac_pred.append(event.value)
        all_l2_demographics.append(cur_l2_demo)
        all_vac_predictions.append(cur_vac_pred)

    all_l2_demographics = np.array(all_l2_demographics)
    all_vac_predictions = np.array(all_vac_predictions)

    starting_prediction = []
    min_prediction = []
    for row in range(all_vac_predictions.shape[0]):
        starting_prediction.append(all_vac_predictions[row,0])
        min_prediction.append(np.min(all_vac_predictions[row,:]))

    combined = list(zip(all_names, starting_prediction, min_prediction))

    # Sort by the first element of each tuple (i.e., values from A)
    combined.sort(key=lambda x: x[2])

    # Unzip back into separate lists
    names_sorted, starting_sorted, min_sorted = zip(*combined)

    # Convert back to lists if needed
    names_sorted = list(names_sorted)
    starting_sorted = list(starting_sorted)
    min_sorted = list(min_sorted)
    
    results_dir = {}
    for ii, name_tup in enumerate(names_sorted):
        for nt in name_tup:
            if nt not in results_dir:
                results_dir[nt] = [min_sorted[ii]]
            else:
                results_dir[nt].append(min_sorted[ii])
    print("Mean min: ", np.mean(min_sorted))
    print("")
    print("Lower than mean:")
    for var in results_dir:
        if np.mean(results_dir[var]) < np.mean(min_sorted):
            print(var, np.mean(results_dir[var]))
    print("")

    print("Higher than mean:")
    for var in results_dir:
        if np.mean(results_dir[var]) > np.mean(min_sorted):
            print(var, np.mean(results_dir[var]))

    print("")

    x = 5
    for ii in range(x):
        print(names_sorted[ii], min_sorted[ii])

for runs_path in file_paths:
    print_stats(runs_path)

In [ ]:
#testing dataset creation
EPOCHS = 100
DISC_LR = 1e-5
GT_LIMIT = 100 #25000
BIAS_LIMIT = 100 #1000
BATCH_SIZE =16
SUBSET_SIZE = 64
week='29'

seed=0
rngs = set_seed(seed,
                device,
                data_init=[True,True],
                data_gen =True,
                network_init=True,)

D1_rngs = copy.deepcopy(rngs)
D1_rngs['seed_bias'] = 359556
D2_rngs = copy.deepcopy(rngs)
D2_rngs['seed_bias'] = 280651
#load in both data sets
d = HouseholdPulse_dataset(ground_truth_path='./data/censusHouseholdPulse_data/ipums_cleaned.csv',
                                    bias_path = './data/censusHouseholdPulse_data/pulse_week'+week+'_cleaned.csv',
                                    rngs=rngs,
                                    device=device,
                                    gt_limit = GT_LIMIT,
                                    bias_limit = BIAS_LIMIT, 
                                    )

In [ ]:
#attempting to find trends in the randomness
#plotting events from saved run logs
# Path to the directory where SummaryWriter saved logs
file_paths = ["runs/"]

def print_stats(runs_path):
    run_folders = []
    for name in os.listdir(runs_path):
        if os.path.isdir(os.path.join(runs_path,name)):
            run_folders.append(os.path.join(runs_path,name))

    event_files = []
    for run_folder in run_folders:
        for name in os.listdir(run_folder):
            if os.path.isdir(os.path.join(run_folder,name)):
                continue
            else:
                event_files.append(os.path.join(run_folder,name))

    all_vac_predictions = []
    all_l2_demographics = []
    for i, event_file in enumerate(event_files):
        # Load event accumulator
        ea = event_accumulator.EventAccumulator(event_file)
        ea.Reload()

        # List available tags (scalars, histograms, images, etc.)
        # Read scalar values (e.g., 'loss', 'accuracy')
        prediction_events = ea.Scalars("Vaccine prediction")
        l2_events = ea.Scalars("L2 Demographics")
        #summ = 0
        #for iter, event in enumerate(prediction_events):
        #    print(event, l2_events[iter])
        cur_vac_pred = []
        cur_l2_demo = []
        for event in prediction_events:
            cur_vac_pred.append(event.value)
        for ievent in l2_events:
            cur_l2_demo.append(ievent.value)
        all_vac_predictions.append(cur_vac_pred)
        all_l2_demographics.append(cur_l2_demo)

    
    all_vac_predictions = np.array(all_vac_predictions)
    all_l2_demographics = np.array(all_l2_demographics)

    r_vales = []
    for i in range(1,70):
        start_point = 80
        end_point = np.shape(all_vac_predictions)[1]-i
        #ending points
        ending_vac_pred = all_vac_predictions[:,end_point]
        ending_l2_demopgrahics = all_l2_demographics[:,end_point]

        ending_vac_pred=np.delete(ending_vac_pred, ending_l2_demopgrahics.argmax())
        ending_l2_demopgrahics=np.delete(ending_l2_demopgrahics, ending_l2_demopgrahics.argmax())

        #plt.scatter(x=ending_l2_demopgrahics, y=ending_vac_pred)
        #plt.xlabel("Ending Demographic L2")
        #plt.ylabel("Ending Vac Prediction")
        #plt.show()
        slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(ending_l2_demopgrahics, ending_vac_pred)
        r_vales.append(r_value)
        #print(r_value)
        #print("R value: ", r_value)
        #print("---------------------------------")
        #average from starting point
        #avg_vac_pred = np.mean(all_vac_predictions[:,start_point:],axis=1)
        #avg_l2_demo = np.mean(all_l2_demographics[:,start_point:],axis=1)
        #print(avg_vac_pred)
        #plt.scatter(x=avg_l2_demo, y=avg_vac_pred)
        #plt.xlabel("avg Demographic L2")
        #plt.ylabel("avg Vac Prediction")
        #slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(avg_vac_pred, avg_l2_demo)
        #print("R value: ", r_value)

    plt.plot(r_vales)
for runs_path in file_paths:
    print_stats(runs_path)

In [46]:
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks
#analyzing consistency over different types of randomness
'''
runs_consistency_baseline_seed:359556
runs_consistency_diffDataSeen_seed:359556
runs_consistency_diffNetworkInit_seed:359556
'''
#plotting events from saved run logs
from tensorboard.backend.event_processing import event_accumulator
import os
# Path to the directory where SummaryWriter saved logs
file_paths = ["./runs"]

def plot_2d_runs(data):
    # Compute mean and standard deviation across experiments
    mean = np.mean(data, axis=0)
    std = np.std(data, axis=0)  # Or use std / sqrt(n) for standard error

    # Time steps (x-axis)
    time_steps = np.arange(data.shape[1])

    # Plot
    plt.figure(figsize=(10, 5))
    plt.plot(time_steps, mean, label='Mean Time Series')
    plt.fill_between(time_steps, mean - std, mean + std, alpha=0.3, label='±1 Std Dev')
    plt.xlabel('Time Step')
    plt.ylabel('Value')
    plt.title('Average Time Series with Error Bands')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

def load_runs_as_numpy(runs_path, var_names, filter_by):
    run_folders = []
    for name in os.listdir(runs_path):
        if filter_by not in name:
            continue
        if os.path.isdir(os.path.join(runs_path,name)):
            run_folders.append(os.path.join(runs_path,name))

    event_files = []
    for run_folder in run_folders:
        for name in os.listdir(run_folder):
            if os.path.isdir(os.path.join(run_folder,name)):
                continue
            else:
                event_files.append(os.path.join(run_folder,name))

    all_data = {}
    for i_name, vname in enumerate(var_names):
        all_data[vname] = []
        for i, event_file in enumerate(event_files):
            # Load event accumulator
            ea = event_accumulator.EventAccumulator(event_file)
            ea.Reload()

            # List available tags (scalars, histograms, images, etc.)
            # Read scalar values (e.g., 'loss', 'accuracy')
            prediction_events = ea.Scalars(vname)
            #summ = 0
            #for iter, event in enumerate(prediction_events):
            #    print(event, l2_events[iter])
            cur_pred = []
            for event in prediction_events:
                cur_pred.append(event.value)
            all_data[vname].append(cur_pred)
        all_data[vname] = np.array(all_data[vname])
    return all_data

def print_stats(runs_path):
    all_data = load_runs_as_numpy(runs_path, ['Vaccine prediction', 'l2 norm demo diff'])
    all_vac_predictions = all_data['Vaccine prediction']
    all_l2_demographics = all_data['l2 norm demo diff']
    vac_stds = np.std(all_vac_predictions,axis=0)
    l2_stds = np.std(all_l2_demographics,axis=0)
    print("Consistency of Vac | consistency of L2 (30 trials)")
    print(np.mean(vac_stds),np.mean(l2_stds))
    print("")

    print("Lowest point of vacc variance")
    print(np.argmin(vac_stds[25:]), np.min(vac_stds[25:]), np.mean(all_vac_predictions[:,np.argmin(vac_stds[25:])]))

    #plot_2d_runs(all_vac_predictions[:,25:])

def find_peaks_troughs(runs_path, filter_by):
    sigma = 25  # smoothing level
    mr = 30
    out = load_runs_as_numpy(runs_path,  
                             ['Vaccine prediction total', 'GenLoss', 'Gen Entropy'],
                             filter_by=filter_by)
    vpt = out['Vaccine prediction total']
    gloss = out['GenLoss']
    entropy = out['Gen Entropy']
    for i, y in enumerate(gloss):
        # Smooth the signal
        smoothed = gaussian_filter1d(y, sigma=sigma)
            # Find peaks and troughs
        peaks, _ = find_peaks(smoothed)
        troughs, _ = find_peaks(-smoothed)

        converted_peaks = [int(p/2) for p in peaks]
        converted_troughs = [int(t/2) for t in troughs]
        #print(vpt.shape, converted_peaks, converted_troughs)
        peak_means = [np.mean(vpt[i,max(cp-mr,0):min(cp+mr,vpt.shape[1])]) for cp in converted_peaks]
        trough_means = [np.mean(vpt[i,max(ct-mr,0):min(ct+mr,vpt.shape[1])]) for ct in converted_troughs]

        print(np.mean(vpt[i, :]), trough_means)

def test(runs_path, filter_by):
    pass

for runs_path in file_paths:
    find_peaks_troughs(runs_path, filter_by='lambdaw:40')

0.6871224122388022 [0.6551445990800857, 0.6576154102881749, 0.7156979193290075]
0.6729444260256631 [0.598505754272143, 0.6732307374477386, 0.6966837058464687]
0.6798268846103124 [0.6694813996553421, 0.7205525418122609]
0.6763100164277213 [0.6486486156781515, 0.6182001809279124, 0.7120959271987279]
0.7205026004995618 [0.7478420873483022, 0.7048376778761546, 0.7184057354927063]
0.6989681037834713 [0.7437829236189525, 0.6552205582459768, 0.6726604968309402, 0.6984525461991627]
0.670120347397668 [0.637715674440066, 0.6759049206972122, 0.6853294402360917]
0.703108115707125 [0.7180659880240758, 0.7188136140505473, 0.6730968525012334]
0.7037366531576429 [0.7313003371159236, 0.6655199259519577]
0.6926399675437382 [0.6419207791487376, 0.6940646270910898]


In [ ]:
#computing average vaccination
from tensorboard.backend.event_processing import event_accumulator
import os
# Path to the directory where SummaryWriter saved logs
runs_path = "./runs_aggregate_noLRSchedule/runs_week29/"

run_folders = []
for name in os.listdir(runs_path):
    if os.path.isdir(os.path.join(runs_path,name)):
        run_folders.append(os.path.join(runs_path,name))

event_files = []
for run_folder in run_folders:
    for name in os.listdir(run_folder):
        if os.path.isdir(os.path.join(run_folder,name)):
            continue
        else:
            event_files.append(os.path.join(run_folder,name))

list_of_ending_predictions = []
distances = []
for i, event_file in enumerate(event_files):
    # Load event accumulator
    ea = event_accumulator.EventAccumulator(event_file)
    ea.Reload()

    # List available tags (scalars, histograms, images, etc.)
    # Read scalar values (e.g., 'loss', 'accuracy')
    prediction_events = ea.Scalars("Vaccine prediction")
    l2_events = ea.Scalars("L2 Demographics")
    #summ = 0
    #for iter, event in enumerate(prediction_events):
    #    print(event, l2_events[iter])
    list_of_ending_predictions.append(np.mean(prediction_events[-1].value))
    distances.append(l2_events[-1].value)

weights = 1 / (np.array(distances) + 1e-10)
weights /= weights.sum()
print(np.average(list_of_ending_predictions,weights=weights))
print(np.average(list_of_ending_predictions), np.std(list_of_ending_predictions))

In [ ]:
#creating bias xbox and gt
GT_SIZE = 10000
BIAS_SIZE = 5000

test = XBoxDatasetSimulation("./data/gcHouse,7attributes.csv")
def save_new_XBOX_csvs():
    global_GT_dict, global_GT_var_order,global_bias_dict,global_bias_var_order = XBOX_get_GT_and_bias_ratios()

    GT_persons_count = get_all_persons_types_count(GT_SIZE,global_GT_dict)
    BT_persons_count = get_all_persons_types_count(BIAS_SIZE,global_bias_dict)

    GT_sampled_df = XBOX_get_sampled_df(global_GT_var_order,
                                        GT_persons_count,
                                        test.df)
    bias_sampled_df = XBOX_get_sampled_df(global_bias_var_order,
                                        BT_persons_count,
                                        test.df)

    bias_df_normalized, bias_NaN_columns = normalize_df(bias_sampled_df)
    gt_df_normalized, gt_df_NaN_columns = normalize_df(GT_sampled_df)
    all_NaNs_columns = bias_NaN_columns and gt_df_NaN_columns

    clean_NaN_by_col_index(bias_df_normalized,all_NaNs_columns)
    clean_NaN_by_col_index(gt_df_normalized,all_NaNs_columns)

    print(bias_df_normalized.shape, gt_df_normalized.shape)
    #save to CSV files
    bias_df_normalized = bias_df_normalized.sample(frac=1).reset_index(drop=True)
    gt_df_normalized = gt_df_normalized.sample(frac=1).reset_index(drop=True)

    bias_df_normalized.to_csv(BIAS_SAVE_PATH + "XBOX_bias.csv",index=False)
    gt_df_normalized.to_csv(GT_SAVE_PATH + "XBOX_GT.csv",index=False)

save_new_XBOX_csvs()

In [ ]:
#data analysis:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ks_2samp

def analyze_dataset_differences(A, B, feature_names=None):
    """
    Analyzes differences between two datasets A and B.

    Args:
        A (np.ndarray): Dataset A, shape (N, D)
        B (np.ndarray): Dataset B, shape (M, D)
        feature_names (list[str], optional): List of feature names for plotting

    Returns:
        dict: Dictionary of comparison metrics per feature
    """
    assert A.shape[1] == B.shape[1], "Datasets must have the same number of features"
    D = A.shape[1]
    results = {}

    if feature_names is None:
        feature_names = [f"Feature {i}" for i in range(D)]

    for i in range(D):
        feat_A = A[:, i]
        feat_B = B[:, i]
        
        # Basic statistics
        mean_A, std_A = np.mean(feat_A), np.std(feat_A)
        mean_B, std_B = np.mean(feat_B), np.std(feat_B)
        
        # KS-test for distributional difference
        ks_stat, ks_pval = ks_2samp(feat_A, feat_B)

        results[feature_names[i]] = {
            "mean_A": mean_A,
            "mean_B": mean_B,
            "std_A": std_A,
            "std_B": std_B,
            "ks_stat": ks_stat,
            "ks_pval": ks_pval,
        }

        # Plot distributions
        plt.figure(figsize=(6, 4))
        sns.kdeplot(feat_A, label='Dataset A', fill=True)
        sns.kdeplot(feat_B, label='Dataset B', fill=True)
        plt.title(f"Distribution of {feature_names[i]}")
        plt.legend()
        plt.tight_layout()
        plt.show()

    return results
GT_LIMIT = 25000
BIAS_LIMIT = 1000
seed=0
rngs = set_seed(seed,
                device,
                data_init=[True,True],
                data_gen =True,
                network_init=True,)
week='29'
D1_rngs = copy.deepcopy(rngs)
D1_rngs['seed_bias'] = 359556
D2_rngs = copy.deepcopy(rngs)
D2_rngs['seed_bias'] = 280651
#load in both data sets
D1 = HouseholdPulse_dataset(ground_truth_path='./data/censusHouseholdPulse_data/ipums_cleaned.csv',
                            bias_path = './data/censusHouseholdPulse_data/pulse_week'+week+'_cleaned.csv',
                            rngs=D1_rngs,
                            device=device,
                            gt_limit = GT_LIMIT,
                            bias_limit = BIAS_LIMIT, 
                            )
D2 = HouseholdPulse_dataset(ground_truth_path='./data/censusHouseholdPulse_data/ipums_cleaned.csv',
                            bias_path = './data/censusHouseholdPulse_data/pulse_week'+week+'_cleaned.csv',
                            rngs=D2_rngs,
                            device=device,
                            gt_limit = GT_LIMIT,
                            bias_limit = BIAS_LIMIT, 
                            )

analyze_dataset_differences(D1.unscaled_biased, D2.unscaled_biased, feature_names=None)

In [ ]:
#deepset classifier helper functions
def sample_dataset(dataset, batch_size, sample_size, rngs, weights=None):
    if weights is None:
        index = np.random.choice(np.arange(len(dataset)), size=sample_size*batch_size, p=weights)
    else:
        index = rngs['np'].choice(np.arange(len(dataset)),size=sample_size*batch_size)
    sampled_data = torch.clone(dataset[index])
    sampled_data = torch.reshape(sampled_data, (batch_size, sample_size, dataset.shape[1]))
    return sampled_data

def binary_accuracy_from_logits(logits, targets, threshold=0.5):
    preds = (logits > threshold).float()
    correct = (preds == targets).sum().item()
    total = targets.size(0)
    return 100.0 * correct / total


In [ ]:
#making new household census data

from HouseholdCensusDataProcessing import * 
from CPSDataProcessing import CPS_recoding_survey_and_census_data
census_df = None
survey_df = None

for w in range(29,30):
    print("processing: ", w)
    week = str(w)
    census_df = pd.read_csv("./data/cps_data/cps_00003.csv")
    survey_df = pd.read_csv("./data/censusHouseholdPulse_data/pulse2021_puf_"+week+".csv")
    survey_df, census_df = CPS_recoding_survey_and_census_data(survey_df, census_df)
    #survey_df.to_csv('./data/cps_data/pulse_week'+week+'_cleaned.csv', index=False)
    #census_df.to_csv('./data/cps_data/ipums_cleaned.csv',index=False)

In [ ]:
#compare dataframes
def compare_dataframes(df1, df2):
    assert list(df1.columns) == list(df2.columns), "Columns must match"

    cols = df1.columns
    n_cols = len(cols)
    n_rows = (n_cols + 2) // 3  # auto-layout: 3 columns per row

    fig, axes = plt.subplots(n_rows, 3, figsize=(15, 5 * n_rows))
    axes = axes.flatten()

    for i, col in enumerate(cols):
        ax = axes[i]
        ax.boxplot([df1[col].dropna(), df2[col].dropna()], labels=["Survey", "Census"])
        ax.set_title(f"Column: {col}")
        ax.grid(True)

    # Hide any unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.show()

compare_dataframes(survey_df.iloc[:, 1:],census_df.iloc[:,1:])